<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **MNPS Job Classification_gpt-4o_Five_Pass (Improved Manager/Coach/Coordinator)**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> # **Version 7.5.7 Changes**
>   - **Improved Pass 5** based on analysis of 527 MNPS job descriptions
>   - Hierarchical decision process: Supervision (3x) → Education (2x) → Customer (1.5x) → Budget (1.5x)
>   - Quantitative thresholds: ≥70% of Essential Functions rule
>   - Data-driven action verb patterns from job analysis
>   - Implicit authority signals for Managers without explicit supervision
>   - Education requirements as key signal (Master's = Coach indicator)
>   - Primary customer framework (Teachers vs Programs vs Teams)
>   - Improved tie-breaker rules for title/duty mismatches
> - **Preserves all v7.5.6 logic**: Five-pass system, all previous improvements
> - **Same GPT-4o-2024-11-20 model**, rate-limiting, and output saving

In [1]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")

    # Optionally extract Ground Truth if present
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
            print("✅ Extracted Ground Truth Masterfile.csv from zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

Mounted at /content/drive
📁 Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251114_205425
📁 Outputs dir: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251114_205425/outputs
📦 Found /content/MNPS Prompt Resources.zip, extracting...
✅ Extracted MNPS Prompt Resources
📄 Batch input: /content/Sample JDs.csv
📄 Ground truth: /content/Ground Truth Masterfile.csv
📄 MNPS roles: /content/MNPS Roles.csv
📄 MNPS KSACs: /content/MNPS KSACs.csv
📄 Competency Extended: /content/Competency Extended Descriptions.csv
📄 Korn Ferry: /content/Korn_Ferry Lominger 38 Competencies.csv
✅ Loaded 43 job descriptions
✅ Loaded 176 ground truth records
✅ Loaded 62 MNPS roles
✅ Loaded 310 MNPS KSACs
✅ Loaded 38 competency descriptions
✅ Loaded 38 Korn Ferry competencies


In [2]:
# ==== 2) Load data and build attribute-only view (ignore title) ====
# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

✅ Built attribute-only view for 43 job descriptions
✅ Ignoring job titles - focusing on job attributes only


In [3]:
# ==== 3) Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic ====
# Get MNPS roles from the loaded data
# Handle different possible column names
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    # Fallback to first column
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# Enhanced closed sets for major and minor role groups
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant'
]
MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Enhanced specialist fallback patterns with Problem Role Cheat Sheet logic
SPECIALIST_FALLBACKS = [
    # Technician patterns - hands-on technical work, equipment, maintenance
    ('Technician', 'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),
    # Analyst patterns - data analysis, research, evaluation
    ('Analyst', 'analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),
    # Teacher patterns - classroom instruction, curriculum, students
    ('Teacher', 'classroom|lesson|instruction|teacher|students|curriculum|teaching|educational|academic'),
    # Coach patterns - mentoring, professional development, instructional support
    ('Coach', 'coach|instructional coach|plc|model lessons|co-teach|mentor|professional development|instructional support'),
    # Clerical Support patterns - administrative, office work, records
    ('Clerical Support', 'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),
    # Counselor patterns - guidance, therapy, mental health
    ('Counselor', 'counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological'),
    # Manager patterns - management, supervision, strategic planning
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager|direct|strategic|planning|policy'),
    # Accountant patterns - financial, accounting, bookkeeping
    ('Accountant', 'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),
    # Coordinator patterns - coordination, organization, facilitation
    ('Coordinator', 'coordinate|organize|facilitate|liaison|program coordination|project coordination|event coordination'),
    # Architect patterns - building/construction vs technology
    ('Architect (Facility-Focused)', 'building|construction|facility|architectural|design|space planning|renovation|infrastructure'),
    ('Architect (Technology-Focused)', 'system|software|technology|IT|database|network|programming|technical architecture')
]

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Enhanced logic to discourage overuse of 'Specialist' based on Problem Role Cheat Sheet."""
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    # Check for more specific role matches first
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    # If no specific match, return Specialist
    return proposed_major

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish between Supervisor and Manager based on education requirements.
    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)
    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'
    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'
    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager based on Problem Role Cheat Sheet."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Coach patterns - instructional support, mentoring, professional development
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    # Manager patterns - strategic planning, policy, budget, supervision
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    # Coordinator patterns - coordination, organization, facilitation
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely "Lead", usually "I", "II", or "III"."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role
    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Check if it's a very senior executive role that might warrant "III"
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'
    return minor_role

print("✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined")

✅ Found 62 MNPS roles
✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined


In [4]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====
def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"
    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)
    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")

    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)
    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")

    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)
    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")

    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")

✅ Built comprehensive KSACs text (37793 characters)
✅ Includes all 4 critical MNPS resource documents


In [5]:


# NEW: Manager/Coach/Coordinator Distinction Prompt for Pass 5 (IMPROVED v7.5.7)
manager_coach_coordinator_prompt = """
You are performing a fifth-pass "Manager vs Coach vs Coordinator" distinction for MNPS job classification.

You will ONLY be asked to review roles classified as "Manager", "Coach", or "Coordinator".
Your job is to determine the BEST fit using a HIERARCHICAL decision process based on analysis of 527 MNPS jobs.

CRITICAL: IGNORE THE JOB TITLE. Focus entirely on actual duties, responsibilities, and authorities.

## HIERARCHICAL DECISION PROCESS (Apply in order, weighted factors)

### STEP 1: SUPERVISION AUTHORITY (Strongest Signal - 3x weight)

**Manager indicators (61% have explicit supervision):**
- Formal supervision: "supervises staff", "conducts performance evaluations", "hires/recommends hiring", "counsels and disciplines", "assigns and reviews work", "approves time and leave"
- Implicit authority (for 64% without explicit supervision): "approves expenditures", "makes spending decisions", "sets priorities for department", "develops strategy", "accountable for function-level results", "leads strategic planning", "responsible for overall management"

**Coordinator indicators (16% have limited supervision):**
- Limited supervision: "may provide work direction", "coordinates work of", "acts as liaison", "serves as lead", "provides guidance" (without evaluation authority)
- Budget monitoring only: "monitors expenditures", "tracks invoices", "prepares budget reports", "reconciles accounts"

**Coach indicators (only 6% have supervision):**
- NO formal supervisory authority
- Influence is relational/expertise-based, not positional
- May "support", "coach", "mentor" but does NOT evaluate performance

**RULE:** Formal supervision → Manager | Limited work direction → Coordinator | No supervision → Coach or Coordinator

### STEP 2: EDUCATION REQUIREMENTS (Strong Signal - 2x weight)

**Coach indicators:** Master's degree required/preferred (67.9% of Coaches require Master's), often in Education/Curriculum, may require teaching certification

**Manager indicators:** Bachelor's degree common (24.3%), degree + management experience, education flexible if experience strong

**Coordinator indicators:** Experience-focused (only 0.7% mention degrees), values certifications/licenses over degrees

**RULE:** Master's required + instructional focus → Strong Coach signal

### STEP 3: PRIMARY CUSTOMER & IMPACT (1.5x weight)

**Coach:** Customer = TEACHERS/STAFF | Impact = INSTRUCTIONAL PRACTICE/STUDENT OUTCOMES | Keywords: "teachers", "students", "classroom", "lesson planning", "instructional strategies", "professional learning" | Scope: thematic (e.g., "math instruction")

**Coordinator:** Customer = PROGRAMS/PROCESSES | Impact = SMOOTH OPERATION/COMPLIANCE | Keywords: "program implementation", "case management", "data entry", "forms", "maintains records", "coordinates services" | Scope: program/project/service area

**Manager:** Customer = TEAM/DEPARTMENT/FUNCTION | Impact = PERFORMANCE/BUDGET/STRATEGIC GOALS | Keywords: "department", "organizational goals", "team performance", "service delivery", "efficiency" | Scope: functional area with KPIs

**RULE:** ≥70% of Essential Functions focus on teachers/instruction → Coach | programs/processes → Coordinator | team/function leadership → Manager

### STEP 4: BUDGET AUTHORITY (1.5x weight)

**Manager:** OWNS budget: "develops and manages budget", "approves expenditures", "allocates resources", "responsible for budget", "makes spending decisions", contract/vendor management

**Coordinator:** TRACKS budget: "monitors grant expenditures", "tracks invoices", "reconciles accounts", "prepares budget reports", "assists with budget"

**Coach:** NO budget authority or minimal: "participates in planning", "provides input", budget language minimal/absent

**RULE:** Develops/approves budget → Manager | Monitors/tracks budget → Coordinator | No budget → Coach

### STEP 5: ACTION VERB PATTERNS (From 527-job data analysis)

**Manager verbs:** "manages" (171x in data), "oversees" (66x), "establishes" (67x), "plans/organizes/directs", "evaluates performance", "develops strategy", "implements policy", "leads", "supervises"

**Coordinator verbs:** "coordinates", "organizes", "schedules", "tracks", "monitors", "collects", "maintains records", "ensures compliance", "supports implementation", "facilitates", "prepares reports"

**Coach verbs:** "assists" (46x in data), "supports" (33x), "collaborates" (30x), "coaches", "models", "facilitates professional learning", "provides feedback", "analyzes data with", "co-plans", "co-teaches"

**RULE:** Count verb frequency by category - highest count suggests role

### STEP 6: NATURE OF WORK

**Coach:** INSTRUCTIONAL/DEVELOPMENTAL - modeling lessons, co-teaching, observing classrooms, facilitating PLCs on instructional practice, analyzing student data, job-embedded learning, coaching cycles

**Coordinator:** OPERATIONAL/PROCESS - managing timelines, tracking deliverables, compliance, data entry, case management, recordkeeping, coordinating services, "trains" through workshops

**Manager:** LEADERSHIP/STRATEGIC - planning/directing function, setting priorities, making decisions, allocating resources, leading change, solving complex problems, performance goals, "develops talent" through supervision

## TIE-BREAKER RULES

**Title "Coordinator" but reads Coach:** IF ≥70% of functions are instructional coaching/PD/teacher support AND no explicit supervision/budget ownership AND Master's required → **Coach**

**Title "Manager" but reads Coordinator:** IF primarily monitors/tracks program, prepares reports, coordinates activities AND lacks staff supervision (no evaluations/hiring/discipline) AND lacks budget ownership → **Coordinator**

**Title "Coach" but mostly operational:** IF majority of functions are case management/logistics/process coordination AND little/no instructional coaching AND no classroom support → **Coordinator**

**Ambiguous Coordinator vs Manager:** Is person accountable for STAFF PERFORMANCE and BUDGET OUTCOMES or just program execution? Staff + budget → **Manager** | Program execution only → **Coordinator**

**Ambiguous Coach vs Manager:** Does role have formal supervisory authority and budget ownership? Yes → **Manager** | No → **Coach** (if instructional)

## EXPERIENCE CONTEXT (for reference, not primary factor)
Coordinators: 5-7 years avg | Managers: 5-6 years avg | Coaches: 3-4 years avg (expertise over tenure)

## FINAL DECISION PROCESS
1. Check supervision (3x weight) - strongest signal
2. Check education (2x weight) - Master's → Coach
3. Check primary customer (1.5x weight) - teachers vs programs vs teams
4. Check budget authority (1.5x weight) - owns vs tracks vs none
5. Check action verbs - frequency counts by category
6. Apply tie-breaker rules if uncertain
7. When truly unclear → Keep current classification

## CRITICAL REMINDERS
- IGNORE JOB TITLE - 64% of Managers don't have "Manager" in title
- Focus on DUTIES not LABELS
- Supervision is strongest signal (3x weight)
- Master's degree = strong Coach signal
- Primary customer: Teachers (Coach) | Programs (Coordinator) | Teams (Manager)
- Percentage threshold: ≥70% rule for primary function
- Conservative: When ambiguous, keep current classification

Return JSON:
{
  "new_job_title": "Updated title if reclassify, otherwise keep",
  "major_role_group": "Manager", "Coach", or "Coordinator",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "STRUCTURED justification including: 
    1. Supervision finding (formal/limited/none)
    2. Education requirement
    3. Primary customer identification  
    4. Budget authority level
    5. Dominant action verb pattern
    6. Which hierarchical step(s) determined classification
    7. Any tie-breaker rules applied
    Cite specific phrases from job description supporting each finding."
}

Constraints: Only "Manager", "Coach", or "Coordinator" | Keep same minor_sub_group unless strong reason | Executive roles rarely "Lead"
"""

print("✅ Manager/Coach/Coordinator distinction prompt for Pass 5 defined (IMPROVED v7.5.7)")
# ==== 5) Enhanced Zero Shot Prompt with Problem Role Cheat Sheet Guidelines ====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.
Process:
- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.
- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.
- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.
- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.
- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.
- Never justify classifications based on job titles - only use job attributes and MNPS standards.
IMPORTANT CLASSIFICATION GUIDELINES (Based on Problem Role Cheat Sheet):
ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Specialist: Specialized knowledge in specific domain, but prefer more specific roles when possible
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting
- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation
  * Manager: Strategic planning, policy development, budget oversight, supervision, management
- **Supervisor vs Manager**:
  * Supervisor: Primarily manages people, no post-high school education required
  * Manager: Does more than manage people, requires minimum associates degree
- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure
  * Architect (Technology-Focused): System/software/IT/database/network/programming
MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III"
- **"Lead"** should be reserved for non-executive roles that lead teams or projects
- **"III"** for very advanced KSACs and senior-level expertise
- **"II"** for intermediate complexity and responsibility
- **"I"** for entry-level or basic complexity
Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level (consider executive role guidelines)
- new_job_title: Should incorporate both major_role_group and minor_sub_group (e.g., "Accountant II", "Facility Coordinator II")
- Provide detailed justification based on job attributes and MNPS KSACs alignment that matches your selected role and level"""

print("✅ Enhanced zero shot prompt with Problem Role Cheat Sheet guidelines defined")

# NEW: Self-Consistency Prompt for Pass 2
self_consistency_prompt = \
"""You previously classified a job description and provided a justification. Now, review your own output for internal consistency.

**Instructions:**
- Compare your stated `major_role_group`, `minor_sub_group`, and `new_job_title` with your `grouping_justification`.
- If the justification **does not logically support** the selected role or level, **correct the classification** to match the reasoning.
- If the justification **supports a different role** (e.g., justification describes coaching but role is "Coordinator"), update the role accordingly.
- **Do not change the justification**—only update the classification fields if they conflict with it.
- Use **only approved MNPS roles** and **minor levels (I, II, III, Lead)**.
- Ensure `new_job_title` reflects the corrected role and level.
- If already consistent, return the original values unchanged.

**Return your response as a JSON object with this exact structure:**
{
  "new_job_title": "...",
  "major_role_group": "...",
  "minor_sub_group": "...",
  "grouping_justification": "..."  // <-- DO NOT MODIFY THIS FIELD
}"""

print("✅ Self-consistency prompt for Pass 2 defined")
# NEW: Manager→Director Promotion Check Prompt for Pass 3
triple_check_prompt = """
You are performing a third-pass "promotion check" for MNPS job classification.

You will ONLY be asked to review roles that were previously classified as "Manager".
Your job is to decide whether the role should remain "Manager" or be elevated to "Director".

Use these MNPS-specific guidelines:

1. Scope of responsibility
   - Manager: Owns a program, team, or sub-unit within an office/department; scope is usually local or departmental.
   - Director: Owns an entire function, office, or district-wide program area (often multiple programs) with system-wide impact.

2. Strategy vs operations
   - Manager: Focuses on implementation, day-to-day operations, and executing strategy set by others.
   - Director: Develops or co-develops strategy, sets direction and priorities, and is responsible for long-term planning.

3. People leadership
   - Manager: Supervises individuals and small teams (specialists, coordinators, technicians).
   - Director: Leads managers and/or multiple teams; provides leadership for an office or functional area.

4. Decision rights, budget, and policy
   - Manager: Implements policies and manages part of a budget within limits set by others.
   - Director: Develops or significantly shapes policies and procedures; owns or co-owns budgets and resource allocation for their function.

5. Accountability and stakeholders
   - Manager: Accountable for performance of a team/program; works mainly with school staff, principals, and department peers.
   - Director: Accountable for district- or system-level outcomes; collaborates with Chiefs, Executive Leadership, and external agencies; often represents MNPS externally.

Upgrade to "Director" ONLY IF the job description clearly shows MOST of the Director characteristics above,
such as district-wide scope, strategy setting, policy/budget ownership, and leadership of other leaders or multiple teams.

If evidence is mixed or ambiguous, KEEP IT AS "Manager".

Important constraints:
- Ignore the original job title text; use duties, responsibilities, scope, and KSAC-related content.
- You may ONLY choose "Manager" or "Director" as major_role_group.
- Keep the minor_sub_group consistent with MNPS conventions (I, II, III, or Lead; executive roles rarely have "Lead").

Return a JSON object with:
{
  "new_job_title": "Updated descriptive title if you upgrade to Director, otherwise keep or lightly refine",
  "major_role_group": "Manager" or "Director",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept Manager or upgraded to Director, citing specific phrases from the job description."
}
"""

print("✅ Manager→Director promotion check prompt for Pass 3 defined")

# NEW: Technician→Skilled Laborer Review Prompt for Pass 4
skilled_laborer_check_prompt = """
You are performing a fourth-pass "Technician vs Skilled Laborer" review for MNPS job classification.

You will ONLY be asked to review roles that were previously classified as "Technician".
Your job is to decide whether the role should remain "Technician" or be reclassified as "Skilled Laborer".

Use these MNPS-specific guidelines:

**Technician:**
- Focus on technical systems work requiring diagnostics and specialized technical knowledge
- HVAC systems (heating, ventilation, air conditioning) - technical diagnostics and system troubleshooting
- Network infrastructure, IT systems, audio-visual systems
- Automotive/vehicle diagnostics and repair
- Behavioral health interventions (e.g., Registered Behavior Technician with specialized training in ABA)
- Equipment troubleshooting requiring technical diagnostics and specialized certifications
- Installation and configuration of complex technical systems
- Examples: HVAC Technician, IT Technician, Audio-Visual Technician, Automotive Technician, Behavior Technician

**Skilled Laborer:**
- Focus on traditional trade-based work (plumbing, electrical, carpentry, painting, grounds)
- Physical work using hand tools and power tools to build, move, repair, or maintain physical environment
- Installation, assembly, and repair of physical structures and fixtures (not complex technical systems)
- Examples of trade work: plumbing (pipes, fixtures, drains), electrical wiring and fixtures, carpentry (building/repairing structures), painting, grounds maintenance, furniture assembly/moving, light construction
- May require trade skills and certifications but emphasizes hands-on physical work over technical diagnostics

**Key Distinction:**
- **Technician** = Technical diagnostics, troubleshooting systems, specialized technical knowledge (HVAC diagnostics, IT systems, behavioral interventions)
- **Skilled Laborer** = Traditional trades with physical hands-on work (plumbing, electrical work, carpentry, painting, grounds maintenance)

**Decision Rules:**
- Plumbing work (installing pipes, fixtures, drains) → Reclassify as "Skilled Laborer"
- Electrical work (wiring, installing fixtures) → Reclassify as "Skilled Laborer"  
- Carpentry (building, repairing structures) → Reclassify as "Skilled Laborer"
- HVAC systems (technical diagnostics, system troubleshooting) → Keep as "Technician"
- IT/Network systems → Keep as "Technician"
- Behavioral interventions (RBT, ABA) → Keep as "Technician"
- If evidence is mixed or unclear → Keep as "Technician"

Important constraints:
- Ignore the original job title text; use duties, responsibilities, and work activities.
- You may ONLY choose "Technician" or "Skilled Laborer" as major_role_group.
- Keep the same minor_sub_group (I, II, III, or Lead) unless there is a strong reason to change it.

Return a JSON object with:
{
  "new_job_title": "Updated title if you reclassify to Skilled Laborer, otherwise keep",
  "major_role_group": "Technician" or "Skilled Laborer",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept Technician or reclassified to Skilled Laborer, citing specific work activities from the job description."
}
"""

print("✅ Technician→Skilled Laborer review prompt for Pass 4 defined")

✅ Enhanced zero shot prompt with Problem Role Cheat Sheet guidelines defined
✅ Self-consistency prompt for Pass 2 defined


In [6]:
# ==== 6) OpenAI API Setup with Rate Limiting Protection ====
import os
from google.colab import userdata

# Get API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

# Use GPT-4o-2024-11-20 for stable performance
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            error_str = str(e).lower()
            # Check for rate limiting errors
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    # Exponential backoff with jitter
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                # Non-rate limiting error, raise immediately
                print(f"❌ Non-rate limiting error: {e}")
                raise e
    # This should never be reached, but just in case
    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined with rate limiting protection")

✅ OpenAI client initialized
✅ Using model: gpt-4o-2024-11-20
✅ call_llm_json_with_retry function defined with rate limiting protection


In [9]:
# ==== 7) Five-Pass Batch Processing with Self-Consistency + Manager→Director Triple Check ====
from tqdm import tqdm

# Third-pass promotion check prompt
triple_check_prompt = """
You are performing a third-pass “promotion check” for MNPS job classification.

You will ONLY be asked to review roles that were previously classified as "Manager".
Your job is to decide whether the role should remain "Manager" or be elevated to "Director".

Use these MNPS-specific guidelines:

1. Scope of responsibility
   - Manager: Owns a program, team, or sub-unit within an office/department; scope is usually local or departmental.
   - Director: Owns an entire function, office, or district-wide program area (often multiple programs) with system-wide impact.

2. Strategy vs operations
   - Manager: Focuses on implementation, day-to-day operations, and executing strategy set by others.
   - Director: Develops or co-develops strategy, sets direction and priorities, and is responsible for long-term planning.

3. People leadership
   - Manager: Supervises individuals and small teams (specialists, coordinators, technicians).
   - Director: Leads managers and/or multiple teams; provides leadership for an office or functional area.

4. Decision rights, budget, and policy
   - Manager: Implements policies and manages part of a budget within limits set by others.
   - Director: Develops or significantly shapes policies and procedures; owns or co-owns budgets and resource allocation for their function.

5. Accountability and stakeholders
   - Manager: Accountable for performance of a team/program; works mainly with school staff, principals, and department peers.
   - Director: Accountable for district- or system-level outcomes; collaborates with Chiefs, Executive Leadership, and external agencies; often represents MNPS externally.

Upgrade to "Director" ONLY IF the job description clearly shows MOST of the Director characteristics above,
such as district-wide scope, strategy setting, policy/budget ownership, and leadership of other leaders or multiple teams.

If evidence is mixed or ambiguous, KEEP IT AS "Manager".

Important constraints:
- Ignore the original job title text; use duties, responsibilities, scope, and KSAC-related content.
- You may ONLY choose "Manager" or "Director" as major_role_group.
- Keep the minor_sub_group consistent with MNPS conventions (I, II, III, or Lead; executive roles rarely have "Lead").

Return a JSON object with:
{
  "new_job_title": "Updated descriptive title if you upgrade to Director, otherwise keep or lightly refine",
  "major_role_group": "Manager" or "Director",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Explain why you kept Manager or upgraded to Director, citing specific phrases from the job description."
}
"""

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using:
       - Pass 1: Initial classification
       - Pass 2: Self-consistency check
       - Pass 3: Manager→Director promotion check (only if final_major == 'Manager')
       - Pass 4: Technician→Skilled Laborer review (only if final_major == 'Technician')
       - Pass 5: Manager/Coach/Coordinator distinction (only if final_major in ['Manager', 'Coach', 'Coordinator'])"""

    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    # === PASS 1: Initial Classification ===
    pass1_prompt = f"""{zero_shot_prompt}
Available MNPS Roles: {', '.join(VALID_ROLES)}
{KSACS_TEXT}
Job Description to Classify:
{job_text}
**IMPORTANT**:
- Ignore the job title completely
- Base classification solely on job attributes
- Use only approved MNPS roles and levels (I, II, III, Lead)
- Apply Problem Role Cheat Sheet guidelines
- Avoid overusing "Specialist"
- Distinguish Supervisor vs Manager based on education requirements
- Use Architect (Facility-Focused) for building/construction roles
- Use Architect (Technology-Focused) for system/software roles
- Executive roles (Coordinator, Principal, Director, Manager) rarely have "Lead" minor sub-grouping
- Ensure new_job_title incorporates both major_role_group and minor_sub_group
- Provide detailed justification that aligns with your selected role and level
Return your response as a JSON object with the following structure:
{{
  "new_job_title": "Descriptive title incorporating major_role_group and minor_sub_group",
  "major_role_group": "One of the approved MNPS roles",
  "minor_sub_group": "I, II, III, or Lead (consider executive role guidelines)",
  "grouping_justification": "Detailed explanation based on job attributes and KSACs alignment that matches your selected role and level"
}}"""

    try:
        # Get initial prediction
        pass1_response = call_llm_json_with_retry(pass1_prompt, MODEL_ID)

        # Apply post-processing logic (as in original)
        major_role = pass1_response.get('major_role_group', 'Other')
        minor_role = pass1_response.get('minor_sub_group', 'I')
        justification = pass1_response.get('grouping_justification', 'No justification provided')
        new_job_title = pass1_response.get('new_job_title', f"{major_role} {minor_role}")

        # Apply rule-based corrections
        major_role = discourage_specialist(job_text, major_role)
        major_role = distinguish_supervisor_manager(job_text, major_role)
        major_role = refine_coordinator_coach_manager(job_text, major_role)
        minor_role = normalize_minor(minor_role)
        minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)
        if not new_job_title or new_job_title == 'Unknown':
            new_job_title = f"{major_role} {minor_role}"

        # Reconstruct clean Pass 1 output
        pass1_clean = {
            "new_job_title": new_job_title,
            "major_role_group": major_role,
            "minor_sub_group": minor_role,
            "grouping_justification": justification
        }

        # === PASS 2: Self-Consistency Check ===
        pass2_prompt = f"""{self_consistency_prompt}

**Your Previous Output:**
{json.dumps(pass1_clean, indent=2)}

**Now perform the self-consistency check and return the corrected (or unchanged) JSON.**
"""
        # Call LLM again for consistency check
        pass2_response = call_llm_json_with_retry(pass2_prompt, MODEL_ID)

        # Extract final values (do NOT re-apply post-processing to avoid overriding LLM correction)
        final_major = pass2_response.get('major_role_group', major_role)
        final_minor = pass2_response.get('minor_sub_group', minor_role)
        final_title = pass2_response.get('new_job_title', new_job_title)
        final_justification = pass2_response.get('grouping_justification', justification)  # should be unchanged

        # Re-normalize minor role (in case LLM outputs "1", etc.)
        final_minor = normalize_minor(final_minor)

        # === PASS 3: Manager → Director Triple-Check (ONLY for Manager) ===
        final_final_major = final_major
        final_final_minor = final_minor
        final_final_title = final_title
        final_final_justification = final_justification

        if final_major == "Manager":
            pass3_input = {
                "new_job_title": final_title,
                "major_role_group": final_major,
                "minor_sub_group": final_minor,
                "grouping_justification": final_justification
            }

            pass3_prompt = f"""{triple_check_prompt}

Full Job Description (no title):
{job_text}

Previous Classification (after two-pass check):
{json.dumps(pass3_input, indent=2)}

Now apply the promotion check and return the corrected (or unchanged) JSON.
Remember:
- You may only choose "Manager" or "Director" as major_role_group.
- Upgrade to Director ONLY with strong evidence of Director-level scope, strategy, policy/budget ownership, and leadership of other leaders or multiple teams.
- If evidence is mixed or unclear, keep Manager.
"""
            try:
                pass3_response = call_llm_json_with_retry(pass3_prompt, MODEL_ID)

                proposed_major = pass3_response.get('major_role_group', final_major)
                # Only accept Manager/Director; ignore any other surprise roles
                if proposed_major in ["Manager", "Director"]:
                    final_final_major = proposed_major
                    final_final_minor = normalize_minor(
                        pass3_response.get('minor_sub_group', final_minor)
                    )
                    final_final_title = pass3_response.get('new_job_title', final_title) or final_title
                    final_final_justification = pass3_response.get(
                        'grouping_justification', final_final_justification
                    )
                # else: silently keep Manager if model misbehaves
            except Exception as e3:
                # If triple-check fails, just keep the two-pass result
                print(f"Warning: Pass 3 failed for row {row_idx}: {e3}")

        
        # === PASS 4: Technician → Skilled Laborer Review (ONLY for Technician) ===
        if final_final_major == "Technician":
            pass4_input = {
                "new_job_title": final_final_title,
                "major_role_group": final_final_major,
                "minor_sub_group": final_final_minor,
                "grouping_justification": final_final_justification
            }

            pass4_prompt = f"""{skilled_laborer_check_prompt}

Full Job Description (no title):
{job_text}

Previous Classification (after previous passes):
{json.dumps(pass4_input, indent=2)}

Now apply the Technician vs Skilled Laborer review and return the corrected (or unchanged) JSON.
Remember:
- You may only choose "Technician" or "Skilled Laborer" as major_role_group.
- Reclassify to Skilled Laborer ONLY if the role primarily involves physical, trade-based work with hand/power tools.
- If the role involves technical systems work, diagnostics, or specialized technical knowledge, keep Technician.
- If evidence is mixed or unclear, keep Technician.
"""
            try:
                pass4_response = call_llm_json_with_retry(pass4_prompt, MODEL_ID)

                proposed_major = pass4_response.get('major_role_group', final_final_major)
                # Only accept Technician/Skilled Laborer; ignore any other surprise roles
                if proposed_major in ["Technician", "Skilled Laborer"]:
                    final_final_major = proposed_major
                    final_final_minor = normalize_minor(
                        pass4_response.get('minor_sub_group', final_final_minor)
                    )
                    final_final_title = pass4_response.get('new_job_title', final_final_title) or final_final_title
                    final_final_justification = pass4_response.get(
                        'grouping_justification', final_final_justification
                    )
                # else: silently keep Technician if model misbehaves
            except Exception as e4:
                # If fourth pass fails, just keep the previous result
                print(f"Warning: Pass 4 failed for row {row_idx}: {e4}")

        # Return final result (after optional Pass 3)
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': final_final_title,
            'major_role_group': final_final_major,
            'minor_sub_group': final_final_minor,
            'grouping_justification': final_final_justification,
            'model_used': MODEL_ID
        }

    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions with rate limiting protection
results = []
print("🚀 Starting FIVE-pass batch processing with Self-Consistency + Director + Skilled Laborer + Manager/Coach/Coordinator...")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    # Add small delay between requests to prevent rate limiting
    time.sleep(0.2)

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v757_five_pass.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")

🚀 Starting THREE-pass batch processing with self-consistency + Manager→Director triple check...


Processing jobs:  21%|██        | 9/43 [02:03<09:50, 17.37s/it]

⚠️  Rate limit hit, waiting 1.3 seconds before retry 1/3


Processing jobs: 100%|██████████| 43/43 [12:54<00:00, 18.01s/it]

✅ Processed 43 job descriptions
✅ Saved results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251114_205425/outputs/Job_Classifications_Batch_gpt4o_v754_three_pass.csv


In [10]:
# ==== 8) Generate Summary Statistics and Examples ====
# Load the results
preds = results_df.copy()

# Generate summary statistics
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

# Create summary
summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 'specialist_count', 'executive_lead_count', 'technician_count', 'skilled_laborer_count', 'manager_count', 'coach_count', 'coordinator_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum()),
        int((preds['major_role_group'] == 'Technician').sum()),
        int((preds['major_role_group'] == 'Skilled Laborer').sum()),
        int((preds['major_role_group'] == 'Manager').sum()),
        int((preds['major_role_group'] == 'Coach').sum()),
        int((preds['major_role_group'] == 'Coordinator').sum())
    ]
})
summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v757_five_pass.csv"
summary_stats.to_csv(summary_path, index=False)

# Show examples of classifications
examples = preds[['source_row_index', 'job_title_original', 'new_job_title',
                  'major_role_group', 'minor_sub_group']].head(10)
examples_path = OUTPUTS_DIR / "examples_gpt4o_v757_five_pass.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(summary_stats.to_string(index=False))
print("\n📝 Major Role Distribution:")
print(major_counts.to_string())
print("\n📝 Minor Role Distribution:")
print(minor_counts.to_string())
print("\n📝 Example Classifications:")
print(examples.to_string(index=False))
print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")


📊 Summary Statistics:
              metric  value
          total_rows     43
  unique_major_roles     22
  unique_minor_roles      4
    specialist_count      0
executive_lead_count      0

📝 Major Role Distribution:
major_role_group
Manager                         10
Coach                            4
Instructor                       3
Coordinator                      3
Analyst                          3
Teacher                          2
Accountant                       2
Technician                       2
Assistant                        1
Translator                       1
Assistant Principal              1
Therapist                        1
Administrative Assistant         1
Librarian                        1
Social Worker                    1
Driver                           1
Counselor                        1
Unclassifiable                   1
Director                         1
Architect (Facility-Focused)     1
Liaison                          1
Principal                    

In [ ]:
# ==== 9) Enhanced Quality Check and Validation ====
# Check for alignment issues between justification and selected roles
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()
    # Check if justification mentions the selected role
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })

# Check for job title format consistency
title_format_issues = []
for idx, row in preds.iterrows():
    new_title = str(row['new_job_title'])
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    # Check if job title incorporates both major and minor roles
    if major_role.lower() not in new_title.lower() or minor_role.lower() not in new_title.lower():
        title_format_issues.append({
            'row_index': row['source_row_index'],
            'new_job_title': new_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role
        })

# Check for executive roles with "Lead" minor sub-grouping (should be rare)
executive_lead_issues = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        executive_lead_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })

# Save quality check results
if alignment_issues:
    alignment_df = pd.DataFrame(alignment_issues)
    alignment_path = OUTPUTS_DIR / "alignment_issues_gpt4o_v757_five_pass.csv"
    alignment_df.to_csv(alignment_path, index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues - saved to {alignment_path}")
else:
    print("✅ No alignment issues found")

if title_format_issues:
    title_format_df = pd.DataFrame(title_format_issues)
    title_format_path = OUTPUTS_DIR / "title_format_issues_gpt4o_v757_five_pass.csv"
    title_format_df.to_csv(title_format_path, index=False)
    print(f"⚠️  Found {len(title_format_issues)} title format issues - saved to {title_format_path}")
else:
    print("✅ No title format issues found")

if executive_lead_issues:
    executive_lead_df = pd.DataFrame(executive_lead_issues)
    executive_lead_path = OUTPUTS_DIR / "executive_lead_issues_gpt4o_v757_five_pass.csv"
    executive_lead_df.to_csv(executive_lead_path, index=False)
    print(f"⚠️  Found {len(executive_lead_issues)} executive roles with 'Lead' minor sub-grouping - saved to {executive_lead_path}")
else:
    print("✅ No executive roles with inappropriate 'Lead' minor sub-grouping found")

print("\n✅ Enhanced quality check completed")